In [0]:
%pip install Faker

dbutils.library.restartPython()

In [0]:
import sys
from pathlib import Path

current = Path.cwd()
repo_root = None

for path in [current, *current.parents]:
    if (
        (path / "databricks" / "jobs" / "run_daily_generator.py").exists()
        and
        (path / "databricks" / "ingestion" / "delta_state.py").exists()
    ):
        repo_root = path
        break

if repo_root is None:
    raise RuntimeError("Could not find AI-Analytics-Data-Platform repo root")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print("Repo root:", repo_root)

In [0]:
import importlib
from datetime import date

import databricks.jobs.run_daily_generator as daily_runner

importlib.reload(daily_runner)

SIMULATION_DATE = date(2026, 8, 23)

SOURCE_TEST_SCHEMA = "ai_analytics.daily_runner_test_source"
BRONZE_TEST_SCHEMA = "ai_analytics.daily_runner_test_bronze"

print("Runner loaded")

In [0]:
spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {SOURCE_TEST_SCHEMA}"
)

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {BRONZE_TEST_SCHEMA}"
)

print("Temporary schemas created")

In [0]:
SOURCE_TABLES = [
    "generator_users",
    "generator_user_state",
    "generator_models",
    "generator_devices",
    "generator_subscription_plans",
    "generator_recent_conversations",
    "generator_recent_messages",
    "generator_subscriptions",
    "generator_daily_counts",
]

for table in SOURCE_TABLES:
    source = f"ai_analytics.source.{table}"
    target = f"{SOURCE_TEST_SCHEMA}.{table}"

    spark.sql(f"DROP TABLE IF EXISTS {target}")

    spark.sql(f"""
        CREATE TABLE {target}
        SHALLOW CLONE {source}
    """)

    print(
        f"{table}:",
        spark.table(target).count(),
        "rows"
    )

In [0]:
BRONZE_TABLES = [
    "user_updates",
    "conversations",
    "messages",
    "completions",
    "model_inferences",
    "feedback",
    "errors",
    "subscriptions",
    "purchases",
    "payments",
]

for table in BRONZE_TABLES:
    source = f"ai_analytics.bronze.{table}"
    target = f"{BRONZE_TEST_SCHEMA}.{table}"

    spark.sql(f"DROP TABLE IF EXISTS {target}")

    spark.sql(f"""
        CREATE TABLE {target}
        SHALLOW CLONE {source}
    """)

    print(
        f"{table}:",
        spark.table(target).count(),
        "rows"
    )

In [0]:
from pyspark.sql import functions as F

for table in BRONZE_TABLES:
    df = spark.table(f"{BRONZE_TEST_SCHEMA}.{table}")

    count = (
        df
        .filter(
            F.col("dt")
            == F.lit("2026-08-23").cast("date")
        )
        .count()
    )

    print(f"{table}: Aug 23 rows = {count}")

    assert count == 0

print("\nPASS: temporary Bronze has no Aug 23 data")

In [0]:
dry_run_summary = daily_runner.run_daily_generator(
    spark,
    simulation_date=None,
    seed=42,
    source_schema=SOURCE_TEST_SCHEMA,
    bronze_schema=BRONZE_TEST_SCHEMA,
    dry_run=True,
)

dry_run_summary

In [0]:
assert dry_run_summary["simulation_date"] == SIMULATION_DATE
assert dry_run_summary["mode"] == "DRY_RUN"

assert dry_run_summary["bronze_write_attempted"] is False
assert dry_run_summary["state_update_attempted"] is False

assert dry_run_summary["completed"] is True

print("Clean messages:",
      dry_run_summary["clean_counts"]["messages"])

print("Bronze physical messages:",
      dry_run_summary["bronze_physical_counts"]["messages"])

print("Injected issues:",
      dry_run_summary["injection_issue_count"])

print("\nPASS: full daily runner dry run")
print("PASS: simulation date automatically resolved to Aug 23")
print("PASS: all validations completed")
print("PASS: no writes attempted")

In [0]:
write_summary = daily_runner.run_daily_generator(
    spark,
    simulation_date=SIMULATION_DATE,
    seed=42,
    source_schema=SOURCE_TEST_SCHEMA,
    bronze_schema=BRONZE_TEST_SCHEMA,
    dry_run=False,
)

write_summary

In [0]:
assert write_summary["mode"] == "WRITE"
assert write_summary["bronze_write_attempted"] is True
assert write_summary["state_update_attempted"] is True
assert write_summary["completed"] is True

assert write_summary["post_write_validation"]["status"] == "PASS"

print("PASS: full temporary daily write completed")

In [0]:
for entity in BRONZE_TABLES:
    actual = (
        spark.table(f"{BRONZE_TEST_SCHEMA}.{entity}")
        .filter(
            F.col("dt")
            == F.lit("2026-08-23").cast("date")
        )
        .count()
    )

    expected = write_summary["bronze_physical_counts"][entity]

    print(
        f"{entity}: "
        f"expected={expected:,}, "
        f"actual={actual:,}"
    )

    assert actual == expected

print("\nPASS: temporary Bronze contains exactly the generated physical batches")

In [0]:
daily_counts = (
    spark.table(
        f"{SOURCE_TEST_SCHEMA}.generator_daily_counts"
    )
    .filter(
        F.col("activity_date")
        == F.lit("2026-08-23").cast("date")
    )
    .orderBy("entity")
)

display(daily_counts)

In [0]:
actual_clean_counts = {
    row["entity"]: row["row_count"]
    for row in daily_counts.collect()
}

expected_clean_counts = {
    "user_updates": 41,
    "conversations": 2472,
    "messages": 31256,
    "errors": 747,
}

assert actual_clean_counts == expected_clean_counts

print("PASS: source state advanced using CLEAN records")

In [0]:
window_summary = (
    spark.table(
        f"{SOURCE_TEST_SCHEMA}.generator_daily_counts"
    )
    .groupBy("entity")
    .agg(
        F.min("activity_date").alias("first_date"),
        F.max("activity_date").alias("last_date"),
        F.countDistinct("activity_date").alias("days"),
        F.count("*").alias("rows"),
    )
    .orderBy("entity")
)

display(window_summary)

In [0]:
retry_summary = daily_runner.run_daily_generator(
    spark,
    simulation_date=SIMULATION_DATE,
    seed=42,
    source_schema=SOURCE_TEST_SCHEMA,
    bronze_schema=BRONZE_TEST_SCHEMA,
    dry_run=False,
)

retry_summary

In [0]:
spark.sql(
    "DROP SCHEMA IF EXISTS ai_analytics.daily_runner_test_source CASCADE"
)

spark.sql(
    "DROP SCHEMA IF EXISTS ai_analytics.daily_runner_test_bronze CASCADE"
)

print("Temporary daily-runner test schemas removed")